# 06 - Batch Inference using SageMaker

This notebook performs batch inference using the trained Logistic Regression model.

It includes:
- Loading a pre-trained model artifact from S3 (registered or tar.gz)
- Specifying a batch input CSV (unlabeled data)
- Running a SageMaker batch transform job
- Saving and optionally uploading the predictions
- Notes and guards for local development (this file is written locally but designed to run inside SageMaker)


## Sagemaker Setup

In [1]:
import boto3
import sagemaker
from sagemaker import image_uris
from datetime import datetime

# Setup SageMaker session and roles
session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
region = session.boto_region_name

# Timestamp for naming
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

# Paths
model_artifact_uri = f"s3://{bucket}/diabetes/registry/model.tar.gz"
batch_output_uri = f"s3://{bucket}/diabetes/batch/output/{timestamp}/"

print("Model Artifact:", model_artifact_uri)
print("Output Location:", batch_output_uri)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Model Artifact: s3://sagemaker-us-east-1-380537322556/diabetes/registry/model.tar.gz
Output Location: s3://sagemaker-us-east-1-380537322556/diabetes/batch/output/2025-06-20-21-20-14/


In [10]:
# import os

# # Set to your actual project folder
# project_dir = "/home/sagemaker-user/AAI540_FinalProject"
# os.chdir(project_dir)
# print("Current working directory is now:", os.getcwd())


Current working directory is now: /home/sagemaker-user/AAI540_FinalProject


In [13]:
# import pandas as pd

# df = pd.read_csv("data/X_val.csv", nrows=5)
# print(df.head())


## Create Sagemaker Model

In [2]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data= model_artifact_uri,  
    role=role,
    entry_point="inference.py",  # IMPORTANT!
    framework_version="1.0-1",   # match your training environment if needed
    sagemaker_session= session
)


## Get Validation Data 

In [20]:
import pandas as pd
import pickle
import boto3
import os

# Path setup
local_path = "data/X_val.pkl"
s3_key = "diabetes/data/X_val.pkl"

LOAD_MODE = 'local'

if LOAD_MODE == "s3":
    print("Loading X_val from S3...")
    s3 = boto3.client("s3")
    os.makedirs("data", exist_ok=True)  # Ensure local folder exists
    with open(local_path, "wb") as f:
        s3.download_fileobj(bucket, s3_key, f)

# Load from local file (works for both local and S3 modes)
with open(local_path, "rb") as f:
    X_val = pickle.load(f)

# Create smaller file for faster execution
# X_val = X_val.head(100)
print("Loaded X_val. Shape:", X_val.shape)


Loaded X_val. Shape: (20353, 189)


In [21]:
import os

# Save to CSV
os.makedirs("data", exist_ok=True)
X_val.to_csv("data/X_val.csv", index=False, header=False)

# Upload to S3 in /batch/ folder
batch_input_uri = session.upload_data(
    path="data/X_val.csv",
    bucket=bucket,
    key_prefix="diabetes/batch"
)

print("Uploaded X_val.csv for batch inference.")
print("S3 path:", batch_input_uri)


Uploaded X_val.csv for batch inference.
S3 path: s3://sagemaker-us-east-1-380537322556/diabetes/batch/X_val.csv


## Batch Inference

In [26]:
import contextlib

transformer = model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    assemble_with="Line",             
    output_path=batch_output_uri,
    accept="text/csv"
)

# Ensure log folder exists
os.makedirs("log", exist_ok=True)

# Run batch transform and suppress streaming logs
with open("log/batch_transform_log.txt", "w") as f, contextlib.redirect_stdout(f):
    transformer.transform(
        data=batch_input_uri,
        content_type="text/csv",
        split_type="Line",
        wait=True
    )

print("Batch transform complete.")
print("Predictions saved to:", batch_output_uri)
print("Logs written to 'batch_transform_log.txt'")


# # Create a Transformer from the trained model
# transformer = model.transformer(
#     instance_count=1,
#     instance_type="ml.m5.large",
#     assemble_with="Line",             
#     output_path=batch_output_uri,
#     accept="text/csv"
# )

# # Run the batch transform job and stream logs to the notebook
# transformer.transform(
#     data=batch_input_uri,
#     content_type="text/csv",
#     split_type="Line",
#     wait=True
# )

# print("Batch transform complete.")
# print("Predictions saved to:", batch_output_uri)


INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2025-06-20-22-23-11-918
INFO:sagemaker:Creating transform job with name: sagemaker-scikit-learn-2025-06-20-22-23-12-568


Batch transform complete.
Predictions saved to: s3://sagemaker-us-east-1-380537322556/diabetes/batch/output/2025-06-20-21-20-14/
Logs written to 'batch_transform_log.txt'


## Inspect Predictions

In [27]:
import re
import pandas as pd

def download_batch_output(s3_uri):
    match = re.match(r"s3://([^/]+)/(.+)", s3_uri)
    bucket_name, prefix = match.group(1), match.group(2)

    s3 = boto3.client("s3")
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
    output_file = next(obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".out"))

    s3.download_file(bucket_name, output_file, "log/predictions.out")
    return pd.read_csv("log/predictions.out", header=None)

df_preds = download_batch_output(batch_output_uri)
df_preds.head()


,0
0,0
1,0
2,0
3,0
4,0


In [24]:
# Compare results

# Path setup
local_path = "data/y_val.pkl"
s3_key = "diabetes/data/y_val.pkl"

LOAD_MODE = 'local'

if LOAD_MODE == "s3":
    print("Loading X_val from S3...")
    s3 = boto3.client("s3")
    os.makedirs("data", exist_ok=True)  # Ensure local folder exists
    with open(local_path, "wb") as f:
        s3.download_fileobj(bucket, s3_key, f)

# Load from local file (works for both local and S3 modes)
with open(local_path, "rb") as f:
    y_val = pickle.load(f)

# Create smaller file for faster execution
# y_val = y_val.head(100)
# print("Loaded y_val. Shape:", y_val.shape)

# Reset index for proper alignment
y_val = y_val.reset_index(drop=True)
df_preds = df_preds.reset_index(drop=True)
df_preds.columns = ["y_pred"]

df_eval = pd.DataFrame({
    "y_true": y_val,
    "y_pred": df_preds["y_pred"]
})
df_eval.head()


,y_true,y_pred
0,0,0
1,1,0
2,0,0
3,0,0
4,0,0


In [28]:
# Classification report on validation data

from sklearn.metrics import classification_report

print(classification_report(df_eval["y_true"], df_eval["y_pred"]))


              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18082
           1       0.00      0.00      0.00      2271

    accuracy                           0.89     20353
   macro avg       0.44      0.50      0.47     20353
weighted avg       0.79      0.89      0.84     20353

